In [1]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv('../data/cleaned_attrition.csv')
df.shape

(1470, 32)

In [2]:
# --- Chi-square tests for categorical variables ---
def chi_square_test(df, col, target='Attrition'):
    contingency = pd.crosstab(df[col], df[target])
    chi2, p, dof, expected = stats.chi2_contingency(contingency)
    n = contingency.sum().sum()
    cramers_v = np.sqrt(chi2 / (n * (min(contingency.shape) - 1)))
    return {'variable': col, 'chi2': round(chi2, 2), 'p_value': round(p, 5),
            'cramers_v': round(cramers_v, 3), 'n': n}

cat_results = [chi_square_test(df, col) for col in ['OverTime', 'MaritalStatus', 'BusinessTravel', 'JobRole']]
pd.DataFrame(cat_results).sort_values('cramers_v', ascending=False)

,variable,chi2,p_value,cramers_v,n
0,OverTime,87.56,0.00000,0.244,1470
3,JobRole,86.19,0.00000,0.242,1470
1,MaritalStatus,46.16,0.00000,0.177,1470
2,BusinessTravel,24.18,0.00001,0.128,1470


In [3]:
# --- T-tests for numeric variables ---
def t_test_with_effect_size(df, col, target='Attrition'):
    yes = df[df[target] == 'Yes'][col]
    no = df[df[target] == 'No'][col]
    t_stat, p_val = stats.ttest_ind(yes, no, equal_var=False)  # Welch's t-test, safer default
    pooled_std = np.sqrt((yes.std()**2 + no.std()**2) / 2)
    cohens_d = (yes.mean() - no.mean()) / pooled_std
    return {'variable': col, 't_stat': round(t_stat, 2), 'p_value': round(p_val, 5),
            'cohens_d': round(cohens_d, 3), 'mean_yes': round(yes.mean(), 1), 'mean_no': round(no.mean(), 1)}

num_results = [t_test_with_effect_size(df, col) for col in ['MonthlyIncome', 'DistanceFromHome', 'Age', 'TotalWorkingYears']]
pd.DataFrame(num_results).sort_values('cohens_d')

,variable,t_stat,p_value,cohens_d,mean_yes,mean_no
3,TotalWorkingYears,-7.02,0.00000,-0.484,8.2,11.9
0,MonthlyIncome,-7.48,0.00000,-0.479,4787.1,6832.7
2,Age,-5.83,0.00000,-0.425,33.6,37.6
1,DistanceFromHome,2.89,0.00414,0.209,10.6,8.9


In [4]:
# --- Mann-Whitney U for ordinal variables (more appropriate than t-test for 1-4 Likert scales) ---
# Effect size: rank-biserial correlation, in [-1, 1].
# Negative = the attrition group tends to score LOWER than the group that stayed.
def mannwhitney_test(df, col, target='Attrition'):
    yes = df[df[target] == 'Yes'][col]
    no = df[df[target] == 'No'][col]
    u_stat, p_val = stats.mannwhitneyu(yes, no, alternative='two-sided')
    rank_biserial = 2 * u_stat / (len(yes) * len(no)) - 1
    return {'variable': col, 'u_stat': u_stat, 'p_value': round(p_val, 5),
            'rank_biserial': round(rank_biserial, 3),
            'median_yes': yes.median(), 'median_no': no.median()}

ord_results = [mannwhitney_test(df, col) for col in ['JobSatisfaction', 'EnvironmentSatisfaction', 'WorkLifeBalance']]
pd.DataFrame(ord_results)

,variable,u_stat,p_value,rank_biserial,median_yes,median_no
0,JobSatisfaction,123362.5,0.00008,-0.156,3.0,3.0
1,EnvironmentSatisfaction,124787.0,0.00022,-0.146,3.0,3.0
2,WorkLifeBalance,135709.5,0.04647,-0.071,3.0,3.0


In [5]:
# --- Consolidated summary table ---
summary = pd.concat([
    pd.DataFrame(cat_results).assign(test='chi-square'),
    pd.DataFrame(num_results).assign(test='welch t-test'),
    pd.DataFrame(ord_results).assign(test='mann-whitney')
], ignore_index=True)
summary.to_csv('../outputs/hypothesis_test_results.csv', index=False)
summary

,variable,chi2,p_value,cramers_v,n,test,t_stat,cohens_d,mean_yes,mean_no,u_stat,rank_biserial,median_yes,median_no
0,OverTime,87.56,0.00000,0.244,1470.0,chi-square,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,MaritalStatus,46.16,0.00000,0.177,1470.0,chi-square,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BusinessTravel,24.18,0.00001,0.128,1470.0,chi-square,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,JobRole,86.19,0.00000,0.242,1470.0,chi-square,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,MonthlyIncome,NaN,0.00000,NaN,NaN,welch t-test,-7.48,-0.479,4787.1,6832.7,NaN,NaN,NaN,NaN
5,DistanceFromHome,NaN,0.00414,NaN,NaN,welch t-test,2.89,0.209,10.6,8.9,NaN,NaN,NaN,NaN
6,Age,NaN,0.00000,NaN,NaN,welch t-test,-5.83,-0.425,33.6,37.6,NaN,NaN,NaN,NaN
7,TotalWorkingYears,NaN,0.00000,NaN,NaN,welch t-test,-7.02,-0.484,8.2,11.9,NaN,NaN,NaN,NaN
8,JobSatisfaction,NaN,0.00008,NaN,NaN,mann-whitney,NaN,NaN,NaN,NaN,123362.5,-0.156,3.0,3.0
9,EnvironmentSatisfaction,NaN,0.00022,NaN,NaN,mann-whitney,NaN,NaN,NaN,NaN,124787.0,-0.146,3.0,3.0


In [6]:
df[['TotalWorkingYears', 'MonthlyIncome', 'Age']].corr()

,TotalWorkingYears,MonthlyIncome,Age
TotalWorkingYears,1.000000,0.772893,0.680381
MonthlyIncome,0.772893,1.000000,0.497855
Age,0.680381,0.497855,1.000000


## Takeaways

- **Every candidate driver is "significant" (p < 0.05).** That is exactly why p-values alone are not useful here: with n = 1,470, even weak associations clear the threshold.
- **Effect sizes separate strong drivers from weak ones.** OverTime (Cramér's V ≈ 0.24), JobRole (≈ 0.24) and TotalWorkingYears (Cohen's d ≈ −0.48) are the largest. DistanceFromHome (d ≈ 0.21) and the satisfaction scores (small rank-biserial values) are weak.
- **Income, age and total working years are strongly correlated** (r = 0.50–0.77), so they carry overlapping information. Only one of them should go into the multivariable model.
- These are pairwise, unadjusted comparisons. The Bayesian model in `03_bayesian_model.ipynb` estimates each driver *while holding the others constant*.